# Vision SFT — Qwen3.5 2B (Resmi Pattern)

Resmi `Qwen3_5_(2B)_Vision.ipynb` notebook'una sadık. **Image + text** ile fine-tune.

## Text SFT vs Vision SFT — Farklar

| Konu | Text SFT (Qwen3 4B Instruct) | Vision SFT (Qwen3.5 2B) |
|------|------------------------------|--------------------------|
| Class | `FastLanguageModel` | **`FastVisionModel`** |
| Loss masking | `train_on_responses_only(...)` | yok — collator handle eder |
| Data collator | default | **`UnslothVisionDataCollator`** |
| Data format | `{"text": "..."}` | `{"messages": [{"role":..., "content":[{"type":"image"},...]}]}` |
| `dataset_text_field` | `"text"` | **`""`** |
| `remove_unused_columns` | True (default) | **False** |
| `dataset_kwargs` | yok | **`{"skip_prepare_dataset": True}`** |
| `tokenizer(...)` çağrısı | `tokenizer(text, ...)` | **`tokenizer(image, text, ...)`** |
| Generation | `temp=0.7, top_p=0.8` | **`temp=1.5, min_p=0.1`** |

## Donanım: RTX 5070 Ti 16GB

Qwen3.5 2B Vision için 16-bit LoRA ~10GB sığar. 4-bit istersen `load_in_4bit=True` (ama Hub'da pre-quantized variant yok).

## 1. Setup

In [ ]:
import unsloth                                  # MUTLAKA EN BAŞTA
from unsloth import FastVisionModel               # FastLanguageModel DEĞİL
from unsloth.trainer import UnslothVisionDataCollator   # Vision için ZORUNLU
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. Model — Qwen3.5 2B

Resmi notebook tam olarak `unsloth/Qwen3.5-2B` kullanıyor. **Pre-quantized variant yok** — full precision'tan yükle, gerekirse `load_in_4bit=True` runtime quantize.

### Diğer Vision Model Seçenekleri

Resmi notebook listelediği vision modeller (`fourbit_models`):
- `unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit`
- `unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit`
- `unsloth/Pixtral-12B-2409-bnb-4bit`
- `unsloth/Qwen2-VL-2B-Instruct-bnb-4bit`
- `unsloth/Qwen2-VL-7B-Instruct-bnb-4bit`
- `unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit`

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = False,                  # 16-bit LoRA için False; 4-bit için True
    use_gradient_checkpointing = "unsloth", # Long context için "unsloth" veya True
)

## 3. LoRA — Vision Layer'lar Dahil

Vision SFT'de **tüm `finetune_*` flag'leri True** — vision encoder de tune edilir.

### finetune_* Flag'lerinin Anlamı

| Flag | Ne yapar |
|------|----------|
| `finetune_vision_layers` | Vision encoder (image encoder) LoRA |
| `finetune_language_layers` | Text LM LoRA |
| `finetune_attention_modules` | Attention katmanları LoRA |
| `finetune_mlp_modules` | MLP/FFN katmanları LoRA |

**Pure text task istersen** (yani vision sadece input ama image'ı tune etmen gerekmiyorsa): `finetune_vision_layers=False`.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,    # Vision encoder tune
    finetune_language_layers   = True,    # Text LM tune
    finetune_attention_modules = True,    # Attention tune
    finetune_mlp_modules       = True,    # MLP tune

    r = 16,                                # 8/16/32/64/128
    lora_alpha = 16,                       # alpha == r
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,                    # RSLoRA destekleniyor
    loftq_config = None,                   # LoftQ destekleniyor
    # target_modules = "all-linear",       # opsiyonel
)

## 4. Veri — LaTeX OCR Dataset

Resmi notebook `unsloth/LaTeX_OCR` kullanıyor — matematiksel formül resimleri + LaTeX gösterimi.

Her sample:
- `image`: PIL Image — formül resmi
- `text`: LaTeX string — beklenen output

### Vision Data Format

Standart text format'tan farklı. Conversational ama `content` bir LIST — `text` ve `image` parçaları içeriyor:

```python
{
    "messages": [
        {"role": "user", "content": [
            {"type": "text", "text": "Describe this image."},
            {"type": "image", "image": <PIL.Image>},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": "..."},
        ]},
    ]
}
```

In [ ]:
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")
print(dataset)
print(f"\nIlk ornek anahtarlari: {list(dataset[0].keys())}")

In [ ]:
# Image'i goruntule
dataset[2]["image"]

In [ ]:
# Karsilik gelen LaTeX
print(dataset[2]["text"])

In [ ]:
# Vision conversation format'ina cevir
instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "text",  "text":  instruction},
                {"type": "image", "image": sample["image"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text",  "text": sample["text"]},
            ]},
        ]
    }

# Tüm dataset'i çevir (list comprehension; not Dataset.map for vision)
converted_dataset = [convert_to_conversation(s) for s in dataset]
print(f'Converted: {len(converted_dataset)}')
print(f'\nIlk ornek (mesajlar):')
for m in converted_dataset[0]["messages"]:
    print(f'  role={m["role"]}, content_types={[c["type"] for c in m["content"]]}')

## 5. Eğitim Öncesi Inference

Modeli train etmeden önce bir test örneğine bakalım — ne tahmin ediyor?

In [ ]:
FastVisionModel.for_inference(model)            # Inference modu

image = dataset[2]["image"]
expected = dataset[2]["text"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},                      # Inference'ta sadece placeholder
        {"type": "text", "text": instruction},
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# Image tokenizer'a ILK arg olarak gecer (text_only'den farkli)
inputs = tokenizer(
    image,                                      # PIL.Image
    input_text,                                 # text
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.5,                          # Qwen3.5 Vision önerisi
    min_p = 0.1,
)

print(f'\n\nExpected: {expected}')

## 6. Trainer — Vision SFT'in Zorunlu Config'leri

**Bu config'ler vision SFT için ZORUNLU:**
- `data_collator = UnslothVisionDataCollator(model, tokenizer)`
- `remove_unused_columns = False`
- `dataset_text_field = ""`
- `dataset_kwargs = {"skip_prepare_dataset": True}`
- `max_length = 2048`

Bunlar olmadan vision data parse edilmez.

In [ ]:
FastVisionModel.for_training(model)             # Training moduna geri al

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # ZORUNLU
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,                          # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "vision_outputs",
        report_to = "none",

        # Vision SFT için ZORUNLU üçlü:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

## 7. Train

In [ ]:
# Memory snapshot
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 8. Eğitim Sonrası Inference

Aynı örneği train sonrası deneyelim — beklenti: model artık doğru LaTeX üretiyor.

In [ ]:
FastVisionModel.for_inference(model)            # Inference modu

image = dataset[2]["image"]
expected = dataset[2]["text"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction},
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.5, min_p = 0.1,
)

print(f'\n\nExpected: {expected}')

## 9. Save

Vision modellerde de aynı 4 save yöntemi:
- LoRA adapter (en küçük)
- Merged 16-bit (vLLM)
- GGUF (Ollama/llama.cpp — vision desteği sınırlı)
- Hub push

In [ ]:
# A. LoRA adapter
model.save_pretrained("qwen35_vision_lora")
tokenizer.save_pretrained("qwen35_vision_lora")

# B. Merged 16-bit (vLLM)
# model.save_pretrained_merged(
#     "qwen35_vision_merged",
#     tokenizer,
#     save_method = "merged_16bit",
# )

# C. Hub push
# model.push_to_hub("USER/qwen35_vision_lora", token="hf_xxx")

print("LoRA saved: qwen35_vision_lora/")

## Tips & Tuzaklar

**1. `train_on_responses_only` vision SFT'de KULLANILMAZ.**
- Vision data collator masking'i kendi yapar.
- Eklemeye çalışırsan veri formatı uyumsuzluğu hatası alırsın.

**2. Vision SFT'in 5 zorunlu config:**
```python
data_collator = UnslothVisionDataCollator(model, tokenizer)
remove_unused_columns = False
dataset_text_field = ""
dataset_kwargs = {"skip_prepare_dataset": True}
max_length = 2048
```

**3. Inference'da image tokenizer'a İLK argüman olarak geçer:**
```python
tokenizer(image, text, ...)   # text_only'de tokenizer(text, ...)
```

**4. `for_inference()` ve `for_training()` arasında geçiş:**
Resmi notebook eğitim öncesi `for_inference()` ile test, sonra `for_training()` ile training, sonra tekrar `for_inference()`. Bu pattern'i takip et.

**5. VRAM beklentisi:**
Qwen3.5 2B Vision 16-bit LoRA → ~10-12 GB peak (RTX 5070 Ti rahat). 4-bit istersen ~6-8 GB.

**6. Vision GGUF kısıtlı:**
Ollama/llama.cpp vision modellere full destek vermiyor. Vision için **vLLM** veya HuggingFace Transformers tercih et.

## Sonraki Adım

- **Daha fazla data** — LaTeX_OCR full dataset (her resim ~5KB, ~80K örnek)
- **Domain spesifik** — finansal grafik OCR, Türkçe belge anlama
- **Custom dataset** — kendi image-text çiftlerin
- **`03_thinking_sft.ipynb`** — Qwen3 4B Thinking ile reasoning SFT